# Kazakh ASR audit in Colab

This notebook is a safe, step-by-step workflow for running the ASR audit project in Google Colab without stressing your local laptop.

Checklist:
- install dependencies
- set Hugging Face cache in /content
- run a tiny smoke test
- run real data preparation
- transcribe with Whisper
- evaluate results
- save outputs to Google Drive if needed


In [ ]:
# 1) Mount Drive (optional but recommended)
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# 2) Get the project into the runtime
#
# Repo is private, so an unauthenticated clone fails. Paste a GitHub token
# when prompted (github.com/settings/tokens -> Generate new token (classic)
# -> scope "repo"). getpass hides the input and it is never written to disk
# or to this notebook file.

import os
from getpass import getpass

REPO_URL = 'https://github.com/assemqb/audit.git'
project_dir = '/content/kk-asr-audit'

if not os.path.exists(project_dir):
    token = getpass('GitHub token (repo scope): ')
    auth_url = REPO_URL.replace('https://', f'https://{token}@')
    !git clone "$auth_url" "$project_dir"
    del token, auth_url  # don't keep the token sitting around in memory

os.chdir(project_dir)
print('Project root:', project_dir)


In [ ]:
# 3) Install dependencies in a safe, reproducible order
!pip install -U pip
!pip install -r requirements.txt

# torch/torchcodec are only used to decode FLEURS audio in prepare_data.py
# (a CPU-side task, nothing to do with Whisper's own GPU speed, which comes
# from ctranslate2). Installing from the CPU wheel index avoids a GPU build
# that expects a matching CUDA nvrtc runtime and fails with
# "libnvrtc.so.13: cannot open shared object file" if the versions don't line up.
!pip install --index-url https://download.pytorch.org/whl/cpu torch torchcodec

import os

# Same path prepare_data.py computes on its own (repo_root/.hf_cache) so every
# step shares one cache instead of downloading models/data twice.
hf_cache = os.path.join(project_dir, '.hf_cache')
os.environ['HF_HOME'] = hf_cache
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(hf_cache, 'hub')
os.environ['HF_DATASETS_CACHE'] = os.path.join(hf_cache, 'datasets')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(hf_cache, 'transformers')
os.environ['HF_HUB_DISABLE_XET'] = '1'

print('Dependencies installed and cache configured at', hf_cache)


In [ ]:
# 4) Safe smoke test: tiny data, tiny run
!python src/prepare_data.py --minutes 1 --split test --limit 1
!python src/transcribe.py --model small --lang kk --device auto --compute_type int8
!python src/evaluate.py --hyp results/hyp_small_kk.tsv
print('Smoke test finished.')


In [ ]:
# 5) Full run for the actual audit
# --device auto picks the Colab GPU automatically when the runtime has one
# (Runtime > Change runtime type > T4 GPU), otherwise falls back to CPU.
!python src/prepare_data.py --minutes 12 --split test
!python src/transcribe.py --model large-v3 --lang kk --device auto --compute_type int8_float16
!python src/transcribe.py --model small --lang kk --device auto --compute_type int8
!python src/transcribe.py --model large-v3 --lang none --device auto --compute_type int8_float16
!python src/evaluate.py --hyp results/hyp_large-v3_kk.tsv
print('Full audit finished.')


In [ ]:
# 6) Optional: save results to Drive
# Use if you want the outputs to survive session restarts.

drive_dir = '/content/drive/MyDrive/kk-asr-audit'
!mkdir -p "$drive_dir"
!cp -r /content/kk-asr-audit/data "$drive_dir/"
!cp -r /content/kk-asr-audit/results "$drive_dir/"
print('Results copied to Drive.')
